# Notebook 1: Load the GitHub Gold Standard into MySQL

The [Gold Standard dataset](https://figshare.com/articles/dataset/A_gold_standard_for_polarity_of_emotions_of_software_developers_in_GitHub/11604597?file=21001260) contains 7,122 GitHub pull request and commit comments that were manually annotated by researchers with sentiment polarity labels (positive, negative, or neutral). It was published by Novielli et al. 2020 in ["Can We Use SE-specific Sentiment Analysis Tools in a Cross-Platform Setting?"](https://doi.org/10.1145/3379597.3387446).

**Here's the problem**: this dataset contains no contextual information about a comment. We don't know where the comment came from, who wrote it, or when.

To fix that, we're going to load the Gold Standard CSV into the same MySQL database as the [GHTorrent 2004 dump](https://web.archive.org/web/20150206005357/http://ghtorrent.org/msr14.html). The GHTorrent 2004 dump contains the contextual project and commit data from GitHub (e.g., author, timestamp, etc.). By joining the Gold Standard sentiment CSV to this database, we recover the context that is missing from the CSV. Once both datasets are in the same database, we can JOIN them together using the `comment ID`.

This is the first step in a pipeline that ends with sentiment-labeled comment data that Kaiaulu can use for analysis.

That context enables two types of dataset expansion using Kaiaulu:
1. **Temporal expansion**: re-download the same projects' comments from 2004 through 2025. This captures comments that were posted after the Gold Standard was originally collected
2. **Horizontal expansion**: download additional data sources for those same projects (e.g., source code, version control history) that can be linked back to sentiment labels

### Step 1: Get the data ready

Before running any code in this notebook:

1. Download the [GitHub Gold Standard dataset](https://figshare.com/articles/dataset/A_gold_standard_for_polarity_of_emotions_of_software_developers_in_GitHub/11604597?file=21001260) (`github_gold.csv`).
2. Download the [GHTorrent 2004 MySQL Database Dump](https://web.archive.org/web/20150206005357/http://ghtorrent.org/msr14.html) (use the MySQL dump). Once downloaded, load it into your local MySQL instance
- Option A: MySQL Workbench
  1. Open MySQL Workbench and connect to your local server
  2. Go to Server → Data Import
  3. Select Import from Self-Contained File and choose the downloaded `.sql` file
  4. Under Default Target Schema, type a name for the database (e.g. `github`). Create it first if needed via File → New Query Tab → `CREATE DATABASE github;`
  5. Click 'Start Import' and wait for it to finish (this may take several minutes)

- Option B: terminal
  ```bash
  # Create the database first
  mysql -u root -p -e "CREATE DATABASE github;"

  # Load the dump (replace the path with wherever you saved the file)
  mysql -u root -p github < /path/to/msr14-mysql.sql
  ```

After loading, verify the import worked by running the following in Workbench or the terminal, you should see tables like `projects`, `commits`, `commit_comments`, `pull_requests`, and `users`:

```sql
USE github;
SHOW TABLES;
```

Optional reference: [GHTorrent schema diagram](https://web.archive.org/web/20150206005412/http://ghtorrent.org/relational.html).

### Step 2: Import dependencies

In [20]:
import mysql.connector
import pandas as pd
from sqlalchemy import create_engine, text

### Step 3: Set your MySQL connection details

Update the variables below to match your local MySQL setup. `CSV_PATH` should point to the path you saved `github_gold.csv`.

In [ ]:
MYSQL_HOST     = "localhost"
MYSQL_PORT     = 3306
MYSQL_USER     = "root"
MYSQL_PASSWORD = "ADD_PASSWORD_HERE"
MYSQL_DB       = "github" # name of the database where GHTorrent was loaded

# Path to github_gold.csv on your local machine
CSV_PATH = "PATH_TO/github_gold.csv"

### Step 4: Create the `comment_sentiment` table

Create a fresh table with three columns matching the CSV structure: `ID` (GitHub comment ID), `Polarity` (positive/negative/neutral), and `Text` (the comment body).

The `ID` column is what we use to join each sentiment label to the matching comment in GHTorrent's `commit_comments` and `pull_request_comments` tables.

In [22]:
conn = mysql.connector.connect(
    host=MYSQL_HOST,
    port=MYSQL_PORT,
    user=MYSQL_USER,
    password=MYSQL_PASSWORD,
    database=MYSQL_DB
)
cursor = conn.cursor()

cursor.execute("DROP TABLE IF EXISTS comment_sentiment;")
print("Dropped existing comment_sentiment table (if any).")

cursor.execute("""
    CREATE TABLE comment_sentiment (
        ID      INT          NULL,
        Polarity VARCHAR(256) NULL,
        Text    TEXT         NULL
    );
""")
conn.commit()
print("Created comment_sentiment table.")

Dropped existing comment_sentiment table (if any).
Created comment_sentiment table.


### Step 5: Load the CSV into MySQL

In [23]:
import csv

insert_sql = "INSERT INTO comment_sentiment (ID, Polarity, Text) VALUES (%s, %s, %s)"

rows_inserted = 0
with open(CSV_PATH, newline='', encoding='utf-8') as f:
    reader = csv.reader(f, delimiter=';', quotechar='"')
    next(reader)  # skip header row
    for row in reader:
        if len(row) >= 3:
            cursor.execute(insert_sql, (row[0] or None, row[1] or None, row[2] or None))
            rows_inserted += 1

conn.commit()
print(f"Inserted {rows_inserted} rows into comment_sentiment.")

Inserted 7122 rows into comment_sentiment.


### Step 6: Verify the load

We expect 7,122 rows (one per annotated comment in the Gold Standard dataset).

We also check that each row has a unique `ID`.

In [24]:
engine = create_engine(
    f"mysql+mysqlconnector://{MYSQL_USER}:{MYSQL_PASSWORD}@{MYSQL_HOST}:{MYSQL_PORT}/{MYSQL_DB}"
)

with engine.connect() as con:
    total_rows     = pd.read_sql(text("SELECT COUNT(*) AS total_rows FROM comment_sentiment;"), con)
    distinct_ids   = pd.read_sql(text("SELECT COUNT(DISTINCT ID) AS distinct_ids FROM comment_sentiment;"), con)

total = total_rows['total_rows'].iloc[0]
unique = distinct_ids['distinct_ids'].iloc[0]

print(f"Total rows    : {total}  (expected 7122)")
print(f"Distinct IDs  : {unique}  (expected 7122)")

if total == 7122 and unique == 7122:
    print("PASS: all rows loaded and all IDs are unique.")
else:
    print("WARNING: counts do not match expected values. Re-check CSV path and delimiter.")

Total rows    : 7122  (expected 7122)
Distinct IDs  : 7122  (expected 7122)
PASS: all rows loaded and all IDs are unique.


### Step 7: Check that comment IDs join to GHTorrent

Now let's do a quick sanity check. Do the `ID` values in our new table actually match `comment_id` values in GHTorrent's `commit_comments` table?

If you get zero rows here, the IDs aren't matching up. The most likely cause is that the wrong GHTorrent dump was loaded. Make sure you used the MSR 2014 version linked in Step 1.

In [25]:
query1 = """
SELECT
    s.ID          AS sentiment_id,
    s.Polarity    AS polarity,
    p.name        AS project_name,
    p.url         AS project_url,
    c.sha         AS commit_sha,
    s.Text        AS comment_text
FROM comment_sentiment s
INNER JOIN commit_comments cc ON s.ID = cc.comment_id
INNER JOIN commits         c  ON c.id = cc.commit_id
INNER JOIN projects        p  ON c.project_id = p.id
LIMIT 10;
"""

with engine.connect() as con:
    result1 = pd.read_sql(text(query1), con)

print(f"Rows returned (showing up to 10): {len(result1)}")
result1

Rows returned (showing up to 10): 10


,sentiment_id,polarity,project_name,project_url,commit_sha,comment_text
0,4063186,neutral,jekyll,https://api.github.com/repos/mojombo/jekyll,cb521b7f9a6887051b982a2053cd402ff019594e,No. I still see the wrong twins. * https://gi...
1,3894703,neutral,jquery,https://api.github.com/repos/jquery/jquery,f6e86c3ca4d527d5453a0b5b9591ef38b5d3c000,"Reverted."""
2,1971084,neutral,MaNGOS,https://api.github.com/repos/mangos/MaNGOS,abfc99ef522b8b6353d051a002d026530ec7d253,You can leave a queue while in queue ? (before...
3,1827828,positive,MaNGOS,https://api.github.com/repos/mangos/MaNGOS,915b77339711ec1278ac06ec80d206133bdb427a,"Didn't look at SpellTargetRestrictions XD"""
4,232603,neutral,clojure,https://api.github.com/repos/clojure/clojure,b43bf20e1ba864c817ada237042cfdc8922831c0,Not sure about what kind of line lengths the p...
5,3565454,positive,netty,https://api.github.com/repos/netty/netty,1fee1ef74ed8ac515c19a7f8eebd16f41a37b7b6,@normanmaurer Nice catch ! Did you make the sa...
6,3504879,neutral,netty,https://api.github.com/repos/netty/netty,cfd514d099fb41b2a467ca208fe1334bb04f8f6c,That's why I didn't close after sending the cl...
7,3413199,neutral,netty,https://api.github.com/repos/netty/netty,78d8f05c218cab107255c4dc1a1344aef138d379,Build result for 78d8f05c218cab107255c4dc1a134...
8,3404541,neutral,netty,https://api.github.com/repos/netty/netty,fd0084ecfa254bc5f619f50ec50a8cb8e3cc083e,Why you think using ImmediateEventExecutor is ...
9,2290082,neutral,jquery,https://api.github.com/repos/jquery/jquery,cef044d82ec0d338b2b69756d3ba08692fb80ae4,These are the ones we currently hardcode in Te...
